# 2.4 — Vector Stores

A **vector store** is a database that stores embeddings and lets you search them by similarity.

We'll use **Chroma** — a lightweight, local vector database that runs in Python.

```
Documents
   ↓ embed
Vectors  →  Store in Chroma  →  Query by similarity  →  Return top-k chunks
```

In [ ]:
!pip install langchain langchain-ollama langchain-community chromadb --quiet

## 1. Prepare Documents

In [ ]:
from langchain.schema import Document

docs = [
    Document(page_content='All full-time employees receive 20 days of annual leave per year.', metadata={'section': 'Leave Policy'}),
    Document(page_content='Sick leave is up to 10 days per year with a medical certificate.', metadata={'section': 'Leave Policy'}),
    Document(page_content='Parental leave is 16 weeks fully paid for primary caregivers.', metadata={'section': 'Leave Policy'}),
    Document(page_content='Employees may work remotely up to 3 days per week.', metadata={'section': 'Remote Work'}),
    Document(page_content='Remote workers must be available during core hours: 10am to 3pm.', metadata={'section': 'Remote Work'}),
    Document(page_content='All remote work equipment is provided by the company.', metadata={'section': 'Remote Work'}),
    Document(page_content='Health insurance is provided for all full-time employees and their immediate family.', metadata={'section': 'Benefits'}),
    Document(page_content='A gym membership subsidy of $50 per month is available.', metadata={'section': 'Benefits'}),
    Document(page_content='Employees receive a $1,000 annual learning and development budget.', metadata={'section': 'Benefits'}),
    Document(page_content='Standard working hours are 9am to 5pm, Monday to Friday.', metadata={'section': 'Working Hours'}),
    Document(page_content='Overtime must be pre-approved and will be compensated at 1.5x the hourly rate.', metadata={'section': 'Working Hours'}),
]

print(f'Prepared {len(docs)} documents')

## 2. Create a Chroma Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='llama3.2')

# Create vector store from documents
# This embeds all documents and stores them in Chroma
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory='./chroma_db'  # save to disk
)

print(f'Vector store created with {vectorstore._collection.count()} documents')

## 3. Similarity Search

In [ ]:
query = 'How many vacation days do I get?'
results = vectorstore.similarity_search(query, k=3)

print(f'Query: "{query}"\n')
for i, doc in enumerate(results):
    print(f'Result {i+1} [{doc.metadata["section"]}]:')
    print(f'  {doc.page_content}')
    print()

## 4. Similarity Search with Scores

In [ ]:
query = 'Can I work from home?'
results_with_scores = vectorstore.similarity_search_with_score(query, k=4)

print(f'Query: "{query}"\n')
for doc, score in results_with_scores:
    print(f'Score: {score:.4f}  [{doc.metadata["section"]}]')
    print(f'  {doc.page_content}')
    print()

## 5. Use as a Retriever

A retriever wraps the vector store and integrates into LangChain pipelines.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

results = retriever.invoke('What benefits does the company offer?')

print('Retrieved documents:')
for doc in results:
    print(f'  [{doc.metadata["section"]}] {doc.page_content}')

## 6. Load an Existing Vector Store from Disk

In [ ]:
# Load the persisted store — no need to re-embed!
loaded_store = Chroma(
    persist_directory='./chroma_db',
    embedding_function=embeddings
)

results = loaded_store.similarity_search('overtime pay', k=2)
for doc in results:
    print(f'[{doc.metadata["section"]}] {doc.page_content}')

## Summary

| Concept | Description |
|---------|-------------|
| `Chroma.from_documents()` | Create a vector store from a list of documents |
| `similarity_search()` | Find top-k most similar documents to a query |
| `similarity_search_with_score()` | Same but also returns similarity scores |
| `as_retriever()` | Wrap the vector store for use in LangChain chains |
| `persist_directory` | Save the vector store to disk to avoid re-embedding |